### When a source is not attached

In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage

d:\GenAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
model=init_chat_model("groq:openai/gpt-oss-120b")

In [6]:
## Create messages
messages=[
    SystemMessage("You are a helpful debate assistant. Whatever topic you are asked about you gotta tell in Favour and in against of that. You specializes in debate and discussions."),     
    HumanMessage("Sports")
]

In [7]:
response=model.invoke(messages)
print(response.content)

**Debate Prompt – “Sports”**  
Below is a balanced set of arguments that can be used in a formal debate, a classroom discussion, or a casual conversation. Each side is broken into clear points, with brief explanations and supporting examples. Feel free to expand any point with statistics, anecdotes, or scholarly references as needed.

---

## 🟢 Arguments **In Favor** of Sports  

| # | Claim | Why It Matters / Evidence |
|---|-------|---------------------------|
| **1** | **Physical health benefits** | Regular participation improves cardiovascular fitness, muscle strength, flexibility, and bone density. Studies show active adults have a 30‑40 % lower risk of heart disease and a 20‑30 % lower risk of type‑2 diabetes. |
| **2** | **Mental‑health boost** | Exercise releases endorphins and reduces cortisol, lowering anxiety and depression. Team sports also foster a sense of belonging, which combats loneliness. |
| **3** | **Life‑skill development** | Sports teach discipline, goal‑setting, 

### When a source is attached

In [8]:
## Loading PDF
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader('AI Discussion.pdf')
docs=loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=600,chunk_overlap=100)
documents=text_splitter.split_documents(docs)
documents[:5]

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-03-16T13:50:16+05:30', 'author': 'python-docx', 'moddate': '2026-03-16T13:50:16+05:30', 'source': 'AI Discussion.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='Artificial Intelligence in the Modern World \nArtificial Intelligence (AI) has become one of the most influential technological \ndevelopments of the 21st century.  \nThe concept refers broadly to machines or software systems that are capable of performing \ntasks that normally require  \nhuman intelligence. These tasks include recognizing patterns, understanding language,'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-03-16T13:50:16+05:30', 'author': 'python-docx', 'moddate': '2026-03-16T13:50:16+05:30', 'source': 'AI Discussion.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='solving problems, and making 

In [20]:
## Vector Embedding(through HuggingFace model) And Vector Store
from langchain_community.embeddings import HuggingFaceEmbeddings    #  for converting texts into vectors
from langchain_community.vectorstores import Chroma        # for storing vectors
embeds = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = Chroma.from_documents(documents, embeds)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 383.94it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x00000186B22B34D0>, search_kwargs={})

In [13]:
# Design ChatPrompt Template
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
                                        You are a helpful debate assistant. Whatever topic you are asked about you gotta tell in Favour and in against of that. You specializes in debate and discussions.
                                        <context>
                                        {context}
                                        </context>
                                        Question: {input}
                                        """)

In [14]:
from langchain.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(model,prompt)

In [15]:
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)

In [18]:
response=retrieval_chain.invoke({"input":"AI"})

In [19]:
print(response["answer"])

**AI – A Balanced Debate**

Below are concise, well‑structured arguments **in favor of** and **against** the continued development and deployment of artificial intelligence.  The points are grouped by the four major domains you mentioned: **Technology, Economics, Philosophy, and Public Policy**.

---

## 1. Technology  

### ✅ In Favor  
| Point | Explanation |
|------|-------------|
| **Performance Gains** | AI can process massive data sets and recognize patterns far faster and more accurately than humans (e.g., image‑recognition, natural‑language understanding). |
| **Automation of Dangerous Tasks** | Robots and AI‑controlled systems can operate in hazardous environments—mines, deep‑sea, nuclear plants—reducing human injury and death. |
| **Accelerated Innovation** | Machine‑learning models assist researchers in drug discovery, materials science, and climate modeling, shortening the time‑to‑solution. |
| **Scalability** | Once trained, an AI model can be replicated globally at near‑z